# PeriClock development and stable-feature selection

Paper: Fig. 3b, 3e and the underlying model-development analyses for Fig. 3c–d / Extended Data Fig. 2. Inputs: original 209-donor CSV, 712-donor CSV, 1,133-donor joblib DataFrame, and saved final stable-feature CSV. Resource downloads are separate; repository links and per-file upload status are in RESOURCE_DOWNLOAD.md. Environment: Python/Scanpy, NumPy, pandas, SciPy, scikit-learn, joblib, Matplotlib and seaborn. Outputs: comparison plots, bootstrap frequency and stable-feature tables, internal validation metrics, new fitted estimators for the held-out branches, and final-fit PDF below `OUTPUT_DIR`. No previous notebook is required. **Static preparation only; no training or cohort analysis executed.** External predictions are not regenerated here.

Original source indices below are **zero-based**. Original cell hashes are retained in cell metadata. Outputs were cleared; inclusion does not imply successful execution.

## Required resources and configuration

Three required prepared training matrices and bundled stable-feature selection; [training matrices](../RESOURCE_DOWNLOAD.md#4-training-feature-matrices). The end-to-end AnnData-to-identical-matrix construction is not included here.

Also supply a licensed local Arial font directory (`PERICLOCK_FONT_DIR`); see [other references](../RESOURCE_DOWNLOAD.md#8-other-reference-databases).

Download locations, sizes, SHA256, versions, permissions and publication status are maintained once in [RESOURCE_DOWNLOAD.md](../RESOURCE_DOWNLOAD.md). Individual GEO/GSM sources and sample choices are in the paper’s Supplementary Table 1. Clock models are uploaded to [Hugging Face Models](https://huggingface.co/LeyiZhang1/PeriClock); [Dataset uploads](https://huggingface.co/datasets/LeyiZhang1/PeriClock-data) are in progress. Both repositories are currently private. Consult the resource guide for per-file availability; donor-level public redistribution remains subject to review.

Set `PERICLOCK_RESOURCE_DIR` for external data and `PERICLOCK_MODEL_DIR` for models, or override individual file paths in the next cell. Keep outputs separate from inputs. Start a fresh kernel and run in order.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from within this GitHub checkout.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "periclock").is_dir() and (p / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from within the PeriClock code checkout")
RESOURCE_DIR = Path(os.environ.get("PERICLOCK_RESOURCE_DIR", "~/periclock-resources")).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PERICLOCK_MODEL_DIR", str(RESOURCE_DIR / "models"))).expanduser().resolve()
# Each individual file-path environment variable still overrides these defaults.
FEATURES_200 = Path(os.environ.get("PERICLOCK_FEATURES_200", str(RESOURCE_DIR / "datasets/PeriClock-200s_features_matrix.csv"))).expanduser().resolve()
FEATURES_700 = Path(os.environ.get("PERICLOCK_FEATURES_700", str(RESOURCE_DIR / "datasets/PeriClock-700s_features_matrix.csv"))).expanduser().resolve()
FEATURES_1000 = Path(os.environ.get("PERICLOCK_FEATURES_1000", str(RESOURCE_DIR / "datasets/PeriClock-1000s_PeriClock_features_matrix.pkl"))).expanduser().resolve()
FINAL_STABLE_FEATURES = Path(os.environ.get("PERICLOCK_STABLE_FEATURES", str(PROJECT_ROOT / "resources/features/bootstrap_selected_stable_features.csv"))).expanduser().resolve()
FONT_DIR = Path(os.environ.get("PERICLOCK_FONT_DIR", "/path/to/Arial")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("PERICLOCK_TUTORIAL_OUTPUT", str(PROJECT_ROOT / "outputs/02_periclock_development"))).expanduser().resolve()
# Input files remain read-only; all relative output paths below use this directory.
for input_path in [FEATURES_200, FEATURES_700, FEATURES_1000, FINAL_STABLE_FEATURES]:
    if not input_path.is_file():
        raise FileNotFoundError(f"Set the input path in this cell: {input_path}")
if not FONT_DIR.is_dir():
    raise FileNotFoundError("Set PERICLOCK_FONT_DIR to a directory containing licensed Arial fonts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)

## Nine-regressor comparison (Fig. 3b)

Source: `final/fig3.ipynb`, cells 1, 4, 7–10 and 13. The feature table contains `Age`, `Individual`, subtype proportions ending `_ratio`, and subtype–gene mean log1p(CP10K) expression columns. Use the original tables, not newly normalized substitutes. The original function catches per-model exceptions; inspect the printed failures and confirm all nine models completed before interpreting its comparison. Cell 14 is excluded because it refers to `correlation_df` and `selected_correlation_df`, neither produced by this branch.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
import scanpy as sc
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

In [ ]:
df_features = pd.read_csv(str(FEATURES_200))
df_features

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.stats import pearsonr

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pcc = np.nan
    else:
        pcc = pearsonr(y_true, y_pred)[0]

    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "PCC": pcc
    }

In [ ]:
def build_and_compare_models(
    df_features,
    target_col="Age",
    individual_col="Individual",
    prop_suffix="_ratio",
    test_size=0.2,
    random_state=42,
    variance_threshold=1e-5,
    k_expr=8000,
    n_age_bins=8
):
    data = df_features.copy()

    data[target_col] = pd.to_numeric(
        data[target_col],
        errors="coerce"
    )

    data = data.dropna(
        subset=[target_col]
    ).reset_index(drop=True)

    y = data[target_col].to_numpy(dtype=float)

    if individual_col in data.columns:
        individual_ids = data[individual_col].astype(str).to_numpy()
    else:
        individual_ids = np.arange(len(data)).astype(str)

    feature_cols = [
        col for col in data.columns
        if col not in [target_col, individual_col]
    ]

    prop_cols = [
        col for col in feature_cols
        if col.endswith(prop_suffix)
    ]

    expr_cols = [
        col for col in feature_cols
        if col not in prop_cols
    ]

    print(f"样本数：{len(data)}")
    print(f"细胞比例特征数：{len(prop_cols)}")
    print(f"表达特征数：{len(expr_cols)}")

    # =========================
    # 1. 先拆分样本
    # =========================
    indices = np.arange(len(data))

    try:
        age_bins = pd.qcut(
            pd.Series(y),
            q=n_age_bins,
            labels=False,
            duplicates="drop"
        )

        if age_bins.value_counts().min() < 2:
            age_bins = None

    except ValueError:
        age_bins = None

    train_idx, valid_idx = train_test_split(
        indices,
        test_size=test_size,
        random_state=random_state,
        stratify=age_bins
    )

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    id_train = individual_ids[train_idx]
    id_valid = individual_ids[valid_idx]

    # =========================
    # 2. 提取训练集和验证集
    # =========================
    X_prop_train = data.loc[
        train_idx,
        prop_cols
    ].to_numpy(dtype=np.float32)

    X_prop_valid = data.loc[
        valid_idx,
        prop_cols
    ].to_numpy(dtype=np.float32)

    X_expr_train = data.loc[
        train_idx,
        expr_cols
    ].to_numpy(dtype=np.float32)

    X_expr_valid = data.loc[
        valid_idx,
        expr_cols
    ].to_numpy(dtype=np.float32)

    # =========================
    # 3. 低方差筛选
    # 只在训练集拟合
    # =========================
    vt = VarianceThreshold(
        threshold=variance_threshold
    )

    X_expr_train_vt = vt.fit_transform(
        X_expr_train
    )

    X_expr_valid_vt = vt.transform(
        X_expr_valid
    )

    expr_cols_after_vt = np.asarray(expr_cols)[
        vt.get_support()
    ]

    # =========================
    # 4. 选择与年龄最相关的前k个
    # 只在训练集拟合
    # =========================
    actual_k = min(
        k_expr,
        X_expr_train_vt.shape[1]
    )

    skb = SelectKBest(
        score_func=f_regression,
        k=actual_k
    )

    X_expr_train_selected = skb.fit_transform(
        X_expr_train_vt,
        y_train
    )

    X_expr_valid_selected = skb.transform(
        X_expr_valid_vt
    )

    selected_expr_cols = expr_cols_after_vt[
        skb.get_support()
    ]

    print(f"低方差筛选后表达特征数：{len(expr_cols_after_vt)}")
    print(f"最终保留表达特征数：{len(selected_expr_cols)}")

    # =========================
    # 5. 标准化
    # 只在训练集拟合
    # =========================
    scaler_prop = StandardScaler()
    scaler_expr = StandardScaler()

    X_prop_train_scaled = scaler_prop.fit_transform(
        X_prop_train
    )

    X_prop_valid_scaled = scaler_prop.transform(
        X_prop_valid
    )

    X_expr_train_scaled = scaler_expr.fit_transform(
        X_expr_train_selected
    )

    X_expr_valid_scaled = scaler_expr.transform(
        X_expr_valid_selected
    )

    # =========================
    # 6. 合并最终特征矩阵
    # =========================
    X_train_final = np.hstack([
        X_prop_train_scaled,
        X_expr_train_scaled
    ])

    X_valid_final = np.hstack([
        X_prop_valid_scaled,
        X_expr_valid_scaled
    ])

    final_feature_names = (
        list(prop_cols)
        + list(selected_expr_cols)
    )

    print(f"训练集矩阵：{X_train_final.shape}")
    print(f"验证集矩阵：{X_valid_final.shape}")

    # =========================
    # 7. 定义模型
    # =========================
    models = {
        "Ridge": Ridge(
            alpha=1.0
        ),

        "Lasso": Lasso(
            alpha=0.01,
            max_iter=30000,
            random_state=random_state
        ),

        "ElasticNet": ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=30000,
            random_state=random_state
        ),

        "PLS": PLSRegression(
            n_components=min(
                20,
                X_train_final.shape[0] - 1,
                X_train_final.shape[1]
            ),
            scale=False
        ),

        "Random Forest": RandomForestRegressor(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state
        ),

        "Extra Trees": ExtraTreesRegressor(
            n_estimators=300,
            max_features="sqrt",
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state
        ),

        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=3,
            random_state=random_state
        ),

        "SVR": SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1,
            gamma="scale",
            cache_size=4000
        ),

        "MLP": MLPRegressor(
            hidden_layer_sizes=(128, 64),
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=2000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=50,
            random_state=random_state
        )
    }

    results = []
    prediction_tables = []
    fitted_models = {}

    # =========================
    # 8. 训练和评估
    # =========================
    for model_name, model in models.items():

        print("\n" + "=" * 60)
        print(f"正在训练：{model_name}")

        start_time = time.time()

        try:
            model.fit(
                X_train_final,
                y_train
            )

            train_pred = np.asarray(
                model.predict(X_train_final)
            ).ravel()

            valid_pred = np.asarray(
                model.predict(X_valid_final)
            ).ravel()

            train_metrics = calculate_metrics(
                y_train,
                train_pred
            )

            valid_metrics = calculate_metrics(
                y_valid,
                valid_pred
            )

            runtime_min = (
                time.time() - start_time
            ) / 60

            results.append({
                "Model": model_name,
                "Train_R2": train_metrics["R2"],
                "Train_MAE": train_metrics["MAE"],
                "Train_RMSE": train_metrics["RMSE"],
                "Train_PCC": train_metrics["PCC"],
                "Validation_R2": valid_metrics["R2"],
                "Validation_MAE": valid_metrics["MAE"],
                "Validation_RMSE": valid_metrics["RMSE"],
                "Validation_PCC": valid_metrics["PCC"],
                "Runtime_min": runtime_min
            })

            prediction_tables.append(
                pd.DataFrame({
                    "Individual": id_train,
                    "Age": y_train,
                    "Predicted_Age": train_pred,
                    "Dataset": "Train",
                    "Model": model_name
                })
            )

            prediction_tables.append(
                pd.DataFrame({
                    "Individual": id_valid,
                    "Age": y_valid,
                    "Predicted_Age": valid_pred,
                    "Dataset": "Validation",
                    "Model": model_name
                })
            )

            fitted_models[model_name] = model

            print(
                f"Train R² = {train_metrics['R2']:.3f}"
            )
            print(
                f"Validation R² = {valid_metrics['R2']:.3f}"
            )
            print(
                f"Validation MAE = {valid_metrics['MAE']:.3f}"
            )
            print(
                f"Validation PCC = {valid_metrics['PCC']:.3f}"
            )

        except Exception as error:
            print(f"{model_name}训练失败：{error}")

    results_df = (
        pd.DataFrame(results)
        .sort_values(
            "Validation_R2",
            ascending=False
        )
        .reset_index(drop=True)
    )

    predictions_df = pd.concat(
        prediction_tables,
        ignore_index=True
    )

    preprocessing_objects = {
        "prop_cols": list(prop_cols),
        "expr_cols": list(expr_cols),
        "variance_selector": vt,
        "select_k_best": skb,
        "selected_expr_cols": list(selected_expr_cols),
        "scaler_prop": scaler_prop,
        "scaler_expr": scaler_expr,
        "final_feature_names": final_feature_names,
        "train_indices": train_idx,
        "validation_indices": valid_idx
    }

    split_data = {
        "X_train": X_train_final,
        "X_valid": X_valid_final,
        "y_train": y_train,
        "y_valid": y_valid,
        "id_train": id_train,
        "id_valid": id_valid
    }

    return (
        results_df,
        predictions_df,
        fitted_models,
        preprocessing_objects,
        split_data
    )

In [ ]:
(
    results_df,
    predictions_df,
    fitted_models,
    preprocessing_objects,
    split_data
) = build_and_compare_models(
    df_features=df_features,
    target_col="Age",
    individual_col="Individual",
    prop_suffix="_ratio",
    test_size=0.2,
    random_state=42,
    variance_threshold=1e-5,
    k_expr=8000,
    n_age_bins=8
)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import seaborn as sns

# ============================================================
# 1. 加载 Arial 字体 + 绘图参数
# ============================================================
font_dir = str(FONT_DIR)
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith((".ttf", ".otf")): fm.fontManager.addfont(os.path.join(font_dir, font_file))

mpl.rcParams.update({
    "font.family": "Arial", "font.sans-serif": ["Arial"], "font.size": 13, "font.weight": "bold",
    "axes.titlesize": 16, "axes.titleweight": "bold", "axes.labelsize": 17, "axes.labelweight": "bold",
    "xtick.labelsize": 14, "ytick.labelsize": 15, "axes.linewidth": 1.3,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "mathtext.fontset": "custom", "mathtext.rm": "Arial", "mathtext.it": "Arial:italic", "mathtext.bf": "Arial:bold"
})

sns.set_theme(style="ticks", rc={"font.family": "Arial", "font.weight": "bold", "axes.labelweight": "bold", "axes.titleweight": "bold"})

# ============================================================
# 2. 整理模型名称
# ============================================================
plot_df = results_df.copy()

model_name_map = {
    "Linear Regression": "Linear Regression", "Random Forest": "Random Forest", "Extra Trees": "Extra Trees",
    "Gradient Boosting": "Gradient Boosting", "ElasticNet": "ElasticNet", "Ridge": "Ridge", "Lasso": "Lasso",
    "SVR": "SVR", "MLP": "MLP", "PLS": "PLSR", "PLSRegression": "PLSR", "KNeighbors": "KNN",
    "KNN": "KNN", "PCA Ridge": "PCA + Ridge", "PCA + Ridge": "PCA + Ridge"
}

plot_df["Model_label"] = plot_df["Model"].map(model_name_map).fillna(plot_df["Model"])

# ============================================================
# 3. 按 Validation R² 从高到低排序
# ============================================================
plot_df = plot_df.sort_values("Validation_R2", ascending=False).reset_index(drop=True)
model_order = plot_df["Model_label"].tolist()

# ============================================================
# 4. 配色
# ============================================================
default_color = "#9DBDD7"
en_color = "#D9A21B"
bar_colors = [en_color if model == "ElasticNet" else default_color for model in plot_df["Model_label"]]

# ============================================================
# 5. 创建画布
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11.8, 6.0), gridspec_kw={"width_ratios": [1, 1], "wspace": 0.10})
y_pos = np.arange(len(plot_df))

# ============================================================
# 6. 左图：Validation R²
# ============================================================
ax1 = axes[0]

bars_r2 = ax1.barh(y=y_pos, width=plot_df["Validation_R2"], color=bar_colors, edgecolor="white", linewidth=0.8, height=0.78)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(plot_df["Model_label"])
ax1.invert_yaxis()

ax1.set_title(r"Validation $R^2$", fontsize=17, fontweight="bold", fontname="Arial", pad=10)
ax1.set_xlabel(r"$R^2$", fontsize=18, fontweight="bold", fontname="Arial")
ax1.set_ylabel("")

ax1.axvline(0, color="#555555", linewidth=1.0)

r2_min = plot_df["Validation_R2"].min()
r2_max = plot_df["Validation_R2"].max()
left_limit = min(-0.05, r2_min - 0.10)
right_limit = max(0.55, r2_max + 0.10)
ax1.set_xlim(left_limit, right_limit)

r2_range = right_limit - left_limit

# 数值标注
for bar, value in zip(bars_r2, plot_df["Validation_R2"]):
    y_center = bar.get_y() + bar.get_height() / 2

    if value >= 0:
        x_text = value + r2_range * 0.012
        ha = "left"
        text_color = "#222222"
    else:
        # 负值放在负柱内部，避免和左侧模型名称重叠
        x_text = value / 2
        ha = "center"
        text_color = "#222222"

    ax1.text(x_text, y_center, f"{value:.2f}", va="center", ha=ha, fontsize=14, fontweight="bold", fontname="Arial", color=text_color)

ax1.tick_params(axis="y", length=0, labelsize=15)
ax1.tick_params(axis="x", length=5, width=1.2, labelsize=14)

for label in ax1.get_yticklabels():
    label.set_fontname("Arial"); label.set_fontweight("bold"); label.set_fontsize(15)

for label in ax1.get_xticklabels():
    label.set_fontname("Arial"); label.set_fontweight("bold"); label.set_fontsize(14)

sns.despine(ax=ax1, top=True, right=True, left=False, bottom=False)

# ============================================================
# 7. 右图：Validation MAE
# ============================================================
ax2 = axes[1]

bars_mae = ax2.barh(y=y_pos, width=plot_df["Validation_MAE"], color=bar_colors, edgecolor="white", linewidth=0.8, height=0.78)

ax2.set_yticks(y_pos)
ax2.set_yticklabels([])
ax2.invert_yaxis()

ax2.set_title("Validation MAE", fontsize=17, fontweight="bold", fontname="Arial", pad=10)
ax2.set_xlabel("Years", fontsize=18, fontweight="bold", fontname="Arial")
ax2.set_ylabel("")

mae_max = plot_df["Validation_MAE"].max()
ax2.set_xlim(0, mae_max * 1.22)

for bar, value in zip(bars_mae, plot_df["Validation_MAE"]):
    y_center = bar.get_y() + bar.get_height() / 2
    ax2.text(value + mae_max * 0.015, y_center, f"{value:.2f}", va="center", ha="left", fontsize=14, fontweight="bold", fontname="Arial")

ax2.tick_params(axis="y", length=0)
ax2.tick_params(axis="x", length=5, width=1.2, labelsize=14)

for label in ax2.get_xticklabels():
    label.set_fontname("Arial"); label.set_fontweight("bold"); label.set_fontsize(14)

sns.despine(ax=ax2, top=True, right=True, left=False, bottom=False)

# ============================================================
# 8. 再次强制所有文字 Arial + Bold
# ============================================================
for ax in axes:
    for text in ax.texts:
        text.set_fontname("Arial"); text.set_fontweight("bold")
    ax.xaxis.label.set_fontname("Arial"); ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontname("Arial"); ax.yaxis.label.set_fontweight("bold")
    ax.title.set_fontname("Arial"); ax.title.set_fontweight("bold")
    for label in ax.get_xticklabels():
        label.set_fontname("Arial"); label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontname("Arial"); label.set_fontweight("bold")

# ============================================================
# 9. 调整布局
# ============================================================
plt.subplots_adjust(left=0.22, right=0.98, bottom=0.14, top=0.88, wspace=0.10)

# ============================================================
# 10. 保存
# ============================================================
plt.savefig("validation_R2_MAE_horizontal.pdf", bbox_inches="tight", transparent=True)
plt.savefig("validation_R2_MAE_horizontal.png", dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

## Bootstrap selection and held-out evaluation: 209, 712 and 1,133 donors

Source: cells 16, 18–32, 34–36 and 38–40. The original 80/20 split, 300 bootstrap iterations, frequency **>= 0.7**, scalers and ElasticNet grids are unchanged. These bootstrap branches use the original unstratified split; the nine-regressor comparison above uses age stratification. Model outputs are new files under `OUTPUT_DIR`, never the downloaded released models. Do not run these expensive training sections simply to check installation.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr

In [ ]:
df_all_features = pd.read_csv(str(FEATURES_200))
df_all_features

In [ ]:
age_col="Age"
id_col="Individual"
prop_suffix="_ratio"
test_size=0.2
random_state=42
vt_threshold=1e-5
k_best=8000
bootstrap_n=300
stable_threshold=0.7
l1_ratio_grid=[0.5, 0.7, 0.9, 0.95, 0.99]
final_l1_ratio_grid=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1]
cv_splits=5
output_dir="fig3/iter_model/inte"
model_name="en_model_inte_bootstrap.pkl"

In [ ]:
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# =========================
# 1. 准备 X / y
# =========================
print("Preparing X and y...")
y = df_all_features[age_col].values
X = df_all_features.drop(columns=[age_col])

prop_cols = [col for col in X.columns if col.endswith(prop_suffix)]
expr_cols = [col for col in X.columns if col not in prop_cols and col != id_col]

X_prop = X[prop_cols].copy()
X_expr = X[expr_cols].copy()

In [ ]:
# =========================
# 2. 划分训练集 / 验证集
# =========================
print("Splitting train and test sets...")
X_prop_train, X_prop_test, X_expr_train, X_expr_test, y_train, y_test = train_test_split(
    X_prop,
    X_expr,
    y,
    test_size=test_size,
    random_state=random_state
)

In [ ]:
# =========================
# 3. 训练集上做方差过滤 + SelectKBest
# =========================
print("Performing feature selection on training set...")
vt = VarianceThreshold(threshold=vt_threshold)
X_expr_train_vt = vt.fit_transform(X_expr_train)
X_expr_test_vt = vt.transform(X_expr_test)

k_real = min(k_best, X_expr_train_vt.shape[1])

skb = SelectKBest(score_func=f_regression, k=k_real)
X_expr_train_selected = skb.fit_transform(X_expr_train_vt, y_train)
X_expr_test_selected = skb.transform(X_expr_test_vt)

selected_expr_cols = X_expr_train.columns[vt.get_support()][skb.get_support()]

In [ ]:
# =========================
# 4. 标准化
# =========================
print("Standardizing features...")
scaler_prop = StandardScaler()
scaler_expr = StandardScaler()

X_prop_train_scaled = scaler_prop.fit_transform(X_prop_train)
X_prop_test_scaled = scaler_prop.transform(X_prop_test)

X_expr_train_scaled = scaler_expr.fit_transform(X_expr_train_selected)
X_expr_test_scaled = scaler_expr.transform(X_expr_test_selected)

all_cols = list(prop_cols) + list(selected_expr_cols)

X_train_final = np.hstack([X_prop_train_scaled, X_expr_train_scaled])
X_test_final = np.hstack([X_prop_test_scaled, X_expr_test_scaled])

X_train_final_df = pd.DataFrame(X_train_final, columns=all_cols, index=X_prop_train.index)
X_test_final_df = pd.DataFrame(X_test_final, columns=all_cols, index=X_prop_test.index)

In [ ]:
# =========================
# 5. 初步 ElasticNetCV 选择参数
# =========================
print("Performing initial ElasticNetCV for parameter tuning...")
base_model = ElasticNetCV(
    l1_ratio=l1_ratio_grid,
    cv=cv_splits,
    max_iter=5000,
    n_jobs=-1,
    random_state=random_state
)

base_model.fit(X_train_final_df, y_train)

best_alpha = base_model.alpha_
best_l1 = base_model.l1_ratio_

print("Best alpha:", best_alpha)
print("Best l1_ratio:", best_l1)

In [ ]:
# =========================
# 6. Bootstrap 特征稳定性筛选
# =========================
print("Performing bootstrap for feature stability selection...")
feature_counter = Counter()

for i in range(bootstrap_n):
    X_boot, y_boot = resample(
        X_train_final_df,
        y_train,
        replace=True,
        n_samples=len(y_train),
        random_state=i
    )

    model = ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1,
        max_iter=5000,
        random_state=random_state
    )

    model.fit(X_boot, y_boot)

    selected = X_train_final_df.columns[model.coef_ != 0]

    for f in selected:
        feature_counter[f] += 1

print("Bootstrap finished.")

selection_freq = pd.DataFrame({
    "Feature": list(feature_counter.keys()),
    "Frequency": [feature_counter[f] / bootstrap_n for f in feature_counter]
}).sort_values("Frequency", ascending=False)

selection_freq.to_csv(
    os.path.join(output_dir, "bootstrap_selection_frequency.csv"),
    index=False
)

In [ ]:
# =========================
# 8. 获取稳定特征
# =========================
stable_features = selection_freq[
    selection_freq["Frequency"] >= stable_threshold
].copy()

selected_features = stable_features["Feature"].tolist()

if len(selected_features) == 0:
    raise ValueError(
        f"没有特征达到 stable_threshold={stable_threshold}，建议降低阈值，比如 0.5 或 0.6。"
    )

X_train_selected = X_train_final_df[selected_features]
X_test_selected = X_test_final_df[selected_features]

stable_features.to_csv(
    os.path.join(output_dir, "stable_features.csv"),
    index=False
)

print(f"Stable features number: {len(selected_features)}")


In [ ]:
# =========================
# 9. 用稳定特征重新训练最终模型
# =========================
print("Training final model with stable features...")
cv = KFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=random_state
)

model_reduced = ElasticNetCV(
    l1_ratio=final_l1_ratio_grid,
    alphas=None,
    cv=cv,
    max_iter=5000,
    n_jobs=-1,
    random_state=random_state
)

scores_reduced = cross_val_score(
    model_reduced,
    X_train_selected,
    y_train,
    cv=cv,
    scoring="r2"
)

print(
    f"训练集交叉验证 R²："
    f"{scores_reduced.mean():.3f} ± {scores_reduced.std():.3f}"
)

model_reduced.fit(X_train_selected, y_train)

In [ ]:
# =========================
# 10. 训练集和验证集预测
# =========================
print("Evaluating final model on train and test sets...")
y_train_pred = model_reduced.predict(X_train_selected)
y_test_pred = model_reduced.predict(X_test_selected)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r, train_p = pearsonr(y_train, y_train_pred)
test_r, test_p = pearsonr(y_test, y_test_pred)

metrics = {
    "best_alpha_initial": best_alpha,
    "best_l1_ratio_initial": best_l1,
    "final_alpha": model_reduced.alpha_,
    "final_l1_ratio": model_reduced.l1_ratio_,
    "n_stable_features": len(selected_features),
    "cv_r2_mean_train": scores_reduced.mean(),
    "cv_r2_std_train": scores_reduced.std(),
    "train_r2": train_r2,
    "train_mae": train_mae,
    "train_pearson_r": train_r,
    "train_pearson_p": train_p,
    "test_r2": test_r2,
    "test_mae": test_mae,
    "test_pearson_r": test_r,
    "test_pearson_p": test_p
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(os.path.join(output_dir, "model_metrics.csv"), index=False)

print("Train R²:", round(train_r2, 3))
print("Test R²:", round(test_r2, 3))
print("Train MAE:", round(train_mae, 3))
print("Test MAE:", round(test_mae, 3))
print("Test Pearson r:", round(test_r, 3))

In [ ]:
 # =========================
# 12. 训练集 + 验证集画到同一张图
# =========================
print("Plotting train and test predictions...")

def plot_prediction_train_test(
    y_train,
    y_train_pred,
    y_test,
    y_test_pred,
    title=None,
    test_label="Validation",
    save_path=None,
    show=True
):
    import numpy as np
    import matplotlib.pyplot as plt

    from scipy.stats import pearsonr
    from sklearn.metrics import r2_score, mean_absolute_error

    # =========================
    # 1. 数据整理
    # =========================
    y_train = np.asarray(y_train).flatten()
    y_train_pred = np.asarray(y_train_pred).flatten()
    y_test = np.asarray(y_test).flatten()
    y_test_pred = np.asarray(y_test_pred).flatten()

    # 去除无效值
    train_mask = np.isfinite(y_train) & np.isfinite(y_train_pred)
    test_mask = np.isfinite(y_test) & np.isfinite(y_test_pred)

    y_train = y_train[train_mask]
    y_train_pred = y_train_pred[train_mask]

    y_test = y_test[test_mask]
    y_test_pred = y_test_pred[test_mask]

    # =========================
    # 2. 计算评价指标
    # =========================
    train_r2 = r2_score(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_mad = np.median(np.abs(y_train - y_train_pred))
    train_pcc = pearsonr(y_train, y_train_pred)[0]

    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mad = np.median(np.abs(y_test - y_test_pred))
    test_pcc = pearsonr(y_test, y_test_pred)[0]

    # =========================
    # 3. 论文风格基础设置
    # =========================
    train_color = "#4C78A8"      # 蓝色
    test_color = "#E76F51"       # 橙红
    ideal_color = "#8C8C8C"      # 灰色 y=x
    regression_color = "#CC4C4C" # 深红回归线

    with plt.rc_context({
        "font.family": "Arial",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8.5,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }):

        # 适合论文多面板拼图的比例
        fig, ax = plt.subplots(figsize=(4.5, 4.2))

        # =========================
        # 4. 固定坐标范围（论文统一）
        # =========================
        xmin = 0
        xmax = 120

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(xmin, xmax)
        ax.set_aspect("equal", adjustable="box")

        # 每20岁一个刻度
        ticks = np.arange(0, 121, 20)
        ax.set_xticks(ticks)
        ax.set_yticks(ticks)

        # =========================
        # 5. 绘制散点
        # =========================
        ax.scatter(
            y_train,
            y_train_pred,
            s=22,
            color=train_color,
            alpha=0.55,
            edgecolors="white",
            linewidths=0.25,
            label=f"Train, n={len(y_train)}",
            zorder=2,
            rasterized=True
        )

        ax.scatter(
            y_test,
            y_test_pred,
            s=27,
            color=test_color,
            alpha=0.80,
            edgecolors="white",
            linewidths=0.3,
            label=f"{test_label}, n={len(y_test)}",
            zorder=3,
            rasterized=True
        )

        # =========================
        # 6. 理想线 y=x
        # =========================
        ax.plot(
            [xmin, xmax],
            [xmin, xmax],
            linestyle="--",
            color=ideal_color,
            linewidth=1.2,
            dashes=(4, 3),
            label=r"Ideal line, $y=x$",
            zorder=1
        )

        # =========================
        # 7. Validation/Test回归线
        # =========================
        slope, intercept = np.polyfit(
            y_test,
            y_test_pred,
            deg=1
        )

        x_line = np.linspace(xmin, xmax, 300)
        y_line = slope * x_line + intercept

        ax.plot(
            x_line,
            y_line,
            color=regression_color,
            linewidth=1.8,
            label=f"{test_label} regression",
            zorder=4
        )

        # =========================
        # 8. 指标文本
        # =========================
        metric_text = (
            r"$\bf{Train}$" + "\n"
            + rf"$R^2$ = {train_r2:.2f}" + "\n"
            + f"MAE = {train_mae:.2f}" + "\n"
            + f"MAD = {train_mad:.2f}" + "\n"
            + f"PCC = {train_pcc:.2f}" + "\n\n"
            + rf"$\bf{{{test_label}}}$" + "\n"
            + rf"$R^2$ = {test_r2:.2f}" + "\n"
            + f"MAE = {test_mae:.2f}" + "\n"
            + f"MAD = {test_mad:.2f}" + "\n"
            + f"PCC = {test_pcc:.2f}"
        )

        ax.text(
            0.04,
            0.96,
            metric_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.5,
            linespacing=1.15,
            bbox=dict(
                boxstyle="round,pad=0.35",
                facecolor="white",
                edgecolor="#B5B5B5",
                linewidth=0.6,
                alpha=0.92
            ),
            zorder=10
        )

        # =========================
        # 9. 坐标轴与标题
        # =========================
        ax.set_xlabel("Chronological age (years)")
        ax.set_ylabel("Predicted age (years)")

        if title is not None:
            ax.set_title(
                title,
                fontsize=12,
                fontweight="bold",
                pad=8
            )

        # 去掉顶部和右侧边框
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.spines["left"].set_color("#333333")
        ax.spines["bottom"].set_color("#333333")

        ax.tick_params(
            axis="both",
            direction="out",
            colors="#333333"
        )

        # 使用很弱的辅助网格
        ax.grid(
            True,
            linestyle="-",
            linewidth=0.4,
            alpha=0.16,
            color="#808080"
        )

        ax.set_axisbelow(True)

        # =========================
        # 10. 图例
        # =========================
        legend = ax.legend(
            loc="lower right",
            frameon=False,
            handlelength=2.3,
            handletextpad=0.6,
            borderaxespad=0.6,
            labelspacing=0.5
        )

        plt.tight_layout(pad=0.6)

        # =========================
        # 11. 保存
        # =========================
        if save_path is not None:
            fig.savefig(
                str(OUTPUT_DIR / "fig3/iter_model/inte/ElasticNet_train_validation.pdf"),
                dpi=600,
                bbox_inches="tight",
                facecolor="white"
            )

            fig.savefig(
                str(OUTPUT_DIR / "fig3/iter_model/inte/ElasticNet_train_validation.png"),
                dpi=600,
                bbox_inches="tight",
                facecolor="white"
            )

        if show:
            plt.show()
        else:
            plt.close(fig)

    # 返回评价指标，方便后续整理结果表
    metrics = {
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_MAD": train_mad,
        "Train_PCC": train_pcc,
        f"{test_label}_R2": test_r2,
        f"{test_label}_MAE": test_mae,
        f"{test_label}_MAD": test_mad,
        f"{test_label}_PCC": test_pcc,
        f"{test_label}_Slope": slope,
        f"{test_label}_Intercept": intercept
    }

    return fig, ax, metrics

In [ ]:
fig, ax, prediction_metrics = plot_prediction_train_test(
    y_train=y_train,
    y_train_pred=y_train_pred,
    y_test=y_test,
    y_test_pred=y_test_pred,
    title=None,
    test_label="Validation",
    save_path=str(OUTPUT_DIR / "fig3/iter_model/inte/ElasticNet_train_validation.pdf")
)

In [ ]:
# =========================
# 13. 保存最终模型和预处理器
# =========================
print("Saving final model and preprocessors...")
joblib.dump(
    {
        "vt": vt,
        "skb": skb,
        "scaler_prop": scaler_prop,
        "scaler_expr": scaler_expr,
        "model": model_reduced,
        "prop_cols": prop_cols,
        "expr_cols": expr_cols,
        "selected_expr_cols": list(selected_expr_cols),
        "selected_features": selected_features,
        "stable_features": stable_features,
        "metrics": metrics
    },
    os.path.join(output_dir, model_name)
)

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr


def build_aging_clock_elasticnet_bootstrap(
    df_all_features,
    age_col="Age",
    id_col="Individual",
    prop_suffix="_ratio",
    test_size=0.2,
    random_state=42,
    vt_threshold=1e-5,
    k_best=8000,
    bootstrap_n=300,
    stable_threshold=0.7,
    l1_ratio_grid=[0.5, 0.7, 0.9, 0.95, 0.99],
    final_l1_ratio_grid=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1],
    cv_splits=5,
    output_dir=str(OUTPUT_DIR / "fig3/iter_model/inte_AIDA"),
    model_name="en_model_inte_AIDA_bootstrap.pkl"
):
    os.makedirs(output_dir, exist_ok=True)

    # =========================
    # 1. 准备 X / y
    # =========================
    print("Preparing X and y...")
    y = df_all_features[age_col].values
    X = df_all_features.drop(columns=[age_col])

    prop_cols = [col for col in X.columns if col.endswith(prop_suffix)]
    expr_cols = [col for col in X.columns if col not in prop_cols and col != id_col]

    X_prop = X[prop_cols].copy()
    X_expr = X[expr_cols].copy()

    # =========================
    # 2. 划分训练集 / 验证集
    # =========================
    print("Splitting train and test sets...")
    X_prop_train, X_prop_test, X_expr_train, X_expr_test, y_train, y_test = train_test_split(
        X_prop,
        X_expr,
        y,
        test_size=test_size,
        random_state=random_state
    )

    # =========================
    # 3. 训练集上做方差过滤 + SelectKBest
    # =========================
    print("Performing feature selection on training set...")
    vt = VarianceThreshold(threshold=vt_threshold)
    X_expr_train_vt = vt.fit_transform(X_expr_train)
    X_expr_test_vt = vt.transform(X_expr_test)

    k_real = min(k_best, X_expr_train_vt.shape[1])

    skb = SelectKBest(score_func=f_regression, k=k_real)
    X_expr_train_selected = skb.fit_transform(X_expr_train_vt, y_train)
    X_expr_test_selected = skb.transform(X_expr_test_vt)

    selected_expr_cols = X_expr_train.columns[vt.get_support()][skb.get_support()]

    # =========================
    # 4. 标准化
    # =========================
    print("Standardizing features...")
    scaler_prop = StandardScaler()
    scaler_expr = StandardScaler()

    X_prop_train_scaled = scaler_prop.fit_transform(X_prop_train)
    X_prop_test_scaled = scaler_prop.transform(X_prop_test)

    X_expr_train_scaled = scaler_expr.fit_transform(X_expr_train_selected)
    X_expr_test_scaled = scaler_expr.transform(X_expr_test_selected)

    all_cols = list(prop_cols) + list(selected_expr_cols)

    X_train_final = np.hstack([X_prop_train_scaled, X_expr_train_scaled])
    X_test_final = np.hstack([X_prop_test_scaled, X_expr_test_scaled])

    X_train_final_df = pd.DataFrame(X_train_final, columns=all_cols, index=X_prop_train.index)
    X_test_final_df = pd.DataFrame(X_test_final, columns=all_cols, index=X_prop_test.index)

    # =========================
    # 5. 初步 ElasticNetCV 选择参数
    # =========================
    print("Performing initial ElasticNetCV for parameter tuning...")
    base_model = ElasticNetCV(
        l1_ratio=l1_ratio_grid,
        cv=cv_splits,
        max_iter=5000,
        n_jobs=-1,
        random_state=random_state
    )

    base_model.fit(X_train_final_df, y_train)

    best_alpha = base_model.alpha_
    best_l1 = base_model.l1_ratio_

    print("Best alpha:", best_alpha)
    print("Best l1_ratio:", best_l1)

    # =========================
    # 6. Bootstrap 特征稳定性筛选
    # =========================
    print("Performing bootstrap for feature stability selection...")
    feature_counter = Counter()

    for i in range(bootstrap_n):
        X_boot, y_boot = resample(
            X_train_final_df,
            y_train,
            replace=True,
            n_samples=len(y_train),
            random_state=i
        )

        model = ElasticNet(
            alpha=best_alpha,
            l1_ratio=best_l1,
            max_iter=5000,
            random_state=random_state
        )

        model.fit(X_boot, y_boot)

        selected = X_train_final_df.columns[model.coef_ != 0]

        for f in selected:
            feature_counter[f] += 1

    print("Bootstrap finished.")

    selection_freq = pd.DataFrame({
        "Feature": list(feature_counter.keys()),
        "Frequency": [feature_counter[f] / bootstrap_n for f in feature_counter]
    }).sort_values("Frequency", ascending=False)

    selection_freq.to_csv(
        os.path.join(output_dir, "bootstrap_selection_frequency.csv"),
        index=False
    )

    # =========================
    # 8. 获取稳定特征
    # =========================
    stable_features = selection_freq[
        selection_freq["Frequency"] >= stable_threshold
    ].copy()

    selected_features = stable_features["Feature"].tolist()

    if len(selected_features) == 0:
        raise ValueError(
            f"没有特征达到 stable_threshold={stable_threshold}，建议降低阈值，比如 0.5 或 0.6。"
        )

    X_train_selected = X_train_final_df[selected_features]
    X_test_selected = X_test_final_df[selected_features]

    stable_features.to_csv(
        os.path.join(output_dir, "stable_features.csv"),
        index=False
    )

    print(f"Stable features number: {len(selected_features)}")

    # =========================
    # 9. 用稳定特征重新训练最终模型
    # =========================
    print("Training final model with stable features...")
    cv = KFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=random_state
    )

    model_reduced = ElasticNetCV(
        l1_ratio=final_l1_ratio_grid,
        alphas=None,
        cv=cv,
        max_iter=5000,
        n_jobs=-1,
        random_state=random_state
    )

    scores_reduced = cross_val_score(
        model_reduced,
        X_train_selected,
        y_train,
        cv=cv,
        scoring="r2"
    )

    print(
        f"训练集交叉验证 R²："
        f"{scores_reduced.mean():.3f} ± {scores_reduced.std():.3f}"
    )

    model_reduced.fit(X_train_selected, y_train)

    # =========================
    # 10. 训练集和验证集预测
    # =========================
    print("Evaluating final model on train and test sets...")
    y_train_pred = model_reduced.predict(X_train_selected)
    y_test_pred = model_reduced.predict(X_test_selected)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    train_r, train_p = pearsonr(y_train, y_train_pred)
    test_r, test_p = pearsonr(y_test, y_test_pred)

    metrics = {
        "best_alpha_initial": best_alpha,
        "best_l1_ratio_initial": best_l1,
        "final_alpha": model_reduced.alpha_,
        "final_l1_ratio": model_reduced.l1_ratio_,
        "n_stable_features": len(selected_features),
        "cv_r2_mean_train": scores_reduced.mean(),
        "cv_r2_std_train": scores_reduced.std(),
        "train_r2": train_r2,
        "train_mae": train_mae,
        "train_pearson_r": train_r,
        "train_pearson_p": train_p,
        "test_r2": test_r2,
        "test_mae": test_mae,
        "test_pearson_r": test_r,
        "test_pearson_p": test_p
    }

    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(os.path.join(output_dir, "model_metrics.csv"), index=False)

    print("Train R²:", round(train_r2, 3))
    print("Test R²:", round(test_r2, 3))
    print("Train MAE:", round(train_mae, 3))
    print("Test MAE:", round(test_mae, 3))
    print("Test Pearson r:", round(test_r, 3))


    # =========================
    # 12. 训练集 + 验证集画到同一张图
    # =========================
    print("Plotting train and test predictions...")

    def plot_prediction_train_test(
        y_train,
        y_train_pred,
        y_test,
        y_test_pred,
        title=None,
        test_label="Validation",
        save_path=None,
        show=True
    ):
        import numpy as np
        import matplotlib.pyplot as plt

        from scipy.stats import pearsonr
        from sklearn.metrics import r2_score, mean_absolute_error

        # =========================
        # 1. 数据整理
        # =========================
        y_train = np.asarray(y_train).flatten()
        y_train_pred = np.asarray(y_train_pred).flatten()
        y_test = np.asarray(y_test).flatten()
        y_test_pred = np.asarray(y_test_pred).flatten()

        # 去除无效值
        train_mask = np.isfinite(y_train) & np.isfinite(y_train_pred)
        test_mask = np.isfinite(y_test) & np.isfinite(y_test_pred)

        y_train = y_train[train_mask]
        y_train_pred = y_train_pred[train_mask]

        y_test = y_test[test_mask]
        y_test_pred = y_test_pred[test_mask]

        # =========================
        # 2. 计算评价指标
        # =========================
        train_r2 = r2_score(y_train, y_train_pred)
        train_mae = mean_absolute_error(y_train, y_train_pred)
        train_mad = np.median(np.abs(y_train - y_train_pred))
        train_pcc = pearsonr(y_train, y_train_pred)[0]

        test_r2 = r2_score(y_test, y_test_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_mad = np.median(np.abs(y_test - y_test_pred))
        test_pcc = pearsonr(y_test, y_test_pred)[0]

        # =========================
        # 3. 论文风格基础设置
        # =========================
        train_color = "#4C78A8"      # 蓝色
        test_color = "#E76F51"       # 橙红
        ideal_color = "#8C8C8C"      # 灰色 y=x
        regression_color = "#CC4C4C" # 深红回归线

        with plt.rc_context({
            "font.family": "Arial",
            "font.size": 10,
            "axes.labelsize": 11,
            "axes.titlesize": 12,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.linewidth": 0.8,
            "xtick.major.width": 0.8,
            "ytick.major.width": 0.8,
            "xtick.major.size": 3.5,
            "ytick.major.size": 3.5,
            "pdf.fonttype": 42,
            "ps.fonttype": 42
        }):

            # 适合论文多面板拼图的比例
            fig, ax = plt.subplots(figsize=(4.5, 4.2))

            # =========================
            # 4. 固定坐标范围（论文统一）
            # =========================
            xmin = 0
            xmax = 120

            ax.set_xlim(xmin, xmax)
            ax.set_ylim(xmin, xmax)
            ax.set_aspect("equal", adjustable="box")

            # 每20岁一个刻度
            ticks = np.arange(0, 121, 20)
            ax.set_xticks(ticks)
            ax.set_yticks(ticks)

            # =========================
            # 5. 绘制散点
            # =========================
            ax.scatter(
                y_train,
                y_train_pred,
                s=22,
                color=train_color,
                alpha=0.55,
                edgecolors="white",
                linewidths=0.25,
                label=f"Train, n={len(y_train)}",
                zorder=2,
                rasterized=True
            )

            ax.scatter(
                y_test,
                y_test_pred,
                s=27,
                color=test_color,
                alpha=0.80,
                edgecolors="white",
                linewidths=0.3,
                label=f"{test_label}, n={len(y_test)}",
                zorder=3,
                rasterized=True
            )

            # =========================
            # 6. 理想线 y=x
            # =========================
            ax.plot(
                [xmin, xmax],
                [xmin, xmax],
                linestyle="--",
                color=ideal_color,
                linewidth=1.2,
                dashes=(4, 3),
                label=r"Ideal line, $y=x$",
                zorder=1
            )

            # =========================
            # 7. Validation/Test回归线
            # =========================
            slope, intercept = np.polyfit(
                y_test,
                y_test_pred,
                deg=1
            )

            x_line = np.linspace(xmin, xmax, 300)
            y_line = slope * x_line + intercept

            ax.plot(
                x_line,
                y_line,
                color=regression_color,
                linewidth=1.8,
                label=f"{test_label} regression",
                zorder=4
            )

            # =========================
            # 8. 指标文本
            # =========================
            metric_text = (
                r"$\bf{Train}$" + "\n"
                + rf"$R^2$ = {train_r2:.2f}" + "\n"
                + f"MAE = {train_mae:.2f}" + "\n"
                + f"MAD = {train_mad:.2f}" + "\n"
                + f"PCC = {train_pcc:.2f}" + "\n\n"
                + rf"$\bf{{{test_label}}}$" + "\n"
                + rf"$R^2$ = {test_r2:.2f}" + "\n"
                + f"MAE = {test_mae:.2f}" + "\n"
                + f"MAD = {test_mad:.2f}" + "\n"
                + f"PCC = {test_pcc:.2f}"
            )

            ax.text(
                0.04,
                0.96,
                metric_text,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8.5,
                linespacing=1.15,
                bbox=dict(
                    boxstyle="round,pad=0.35",
                    facecolor="white",
                    edgecolor="#B5B5B5",
                    linewidth=0.6,
                    alpha=0.92
                ),
                zorder=10
            )

            # =========================
            # 9. 坐标轴与标题
            # =========================
            ax.set_xlabel("Chronological age (years)")
            ax.set_ylabel("Predicted age (years)")

            if title is not None:
                ax.set_title(
                    title,
                    fontsize=12,
                    fontweight="bold",
                    pad=8
                )

            # 去掉顶部和右侧边框
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            ax.spines["left"].set_color("#333333")
            ax.spines["bottom"].set_color("#333333")

            ax.tick_params(
                axis="both",
                direction="out",
                colors="#333333"
            )

            # 使用很弱的辅助网格
            ax.grid(
                True,
                linestyle="-",
                linewidth=0.4,
                alpha=0.16,
                color="#808080"
            )

            ax.set_axisbelow(True)

            # =========================
            # 10. 图例
            # =========================
            legend = ax.legend(
                loc="lower right",
                frameon=False,
                handlelength=2.3,
                handletextpad=0.6,
                borderaxespad=0.6,
                labelspacing=0.5
            )

            plt.tight_layout(pad=0.6)

            # =========================
            # 11. 保存
            # =========================
            if save_path is not None:
                fig.savefig(
                    os.path.join(save_path, "ElasticNet_train_validation.png"),
                    dpi=600,
                    bbox_inches="tight",
                    facecolor="white"
                )
                
                fig.savefig(
                    os.path.join(save_path, "ElasticNet_train_validation.pdf"),
                    dpi=600,
                    bbox_inches="tight",
                    facecolor="white"
                )

            if show:
                plt.show()
            else:
                plt.close(fig)

        # 返回评价指标，方便后续整理结果表
        metrics = {
            "Train_R2": train_r2,
            "Train_MAE": train_mae,
            "Train_MAD": train_mad,
            "Train_PCC": train_pcc,
            f"{test_label}_R2": test_r2,
            f"{test_label}_MAE": test_mae,
            f"{test_label}_MAD": test_mad,
            f"{test_label}_PCC": test_pcc,
            f"{test_label}_Slope": slope,
            f"{test_label}_Intercept": intercept
        }

        return fig, ax, metrics

    fig, ax, prediction_metrics = plot_prediction_train_test(
        y_train=y_train,
        y_train_pred=y_train_pred,
        y_test=y_test,
        y_test_pred=y_test_pred,
        title=None,
        test_label="Validation",
        save_path=output_dir
)

    # =========================
    # 13. 保存最终模型和预处理器
    # =========================
    print("Saving final model and preprocessors...")
    joblib.dump(
        {
            "vt": vt,
            "skb": skb,
            "scaler_prop": scaler_prop,
            "scaler_expr": scaler_expr,
            "model": model_reduced,
            "prop_cols": prop_cols,
            "expr_cols": expr_cols,
            "selected_expr_cols": list(selected_expr_cols),
            "selected_features": selected_features,
            "stable_features": stable_features,
            "metrics": metrics
        },
        os.path.join(output_dir, model_name)
    )

    print(f"Model saved to: {os.path.join(output_dir, model_name)}")

    return {
        "model": model_reduced,
        "metrics": metrics_df,
        "selection_freq": selection_freq,
        "stable_features": stable_features,
        "selected_features": selected_features,
        "X_train_selected": X_train_selected,
        "X_test_selected": X_test_selected,
        "y_train": y_train,
        "y_test": y_test,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred
    }

In [ ]:
df_all_features = pd.read_csv(str(FEATURES_700))
df_all_features

In [ ]:
result = build_aging_clock_elasticnet_bootstrap(
    df_all_features=df_all_features,
    age_col="Age",
    id_col="Individual",
    prop_suffix="_ratio",
    test_size=0.2,
    bootstrap_n=300,
    stable_threshold=0.7,
    output_dir=str(OUTPUT_DIR / "fig3/iter_model/inte_AIDA")
)

In [ ]:
df_all_features = joblib.load(str(FEATURES_1000))
df_all_features

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr


def build_aging_clock_elasticnet_bootstrap(
    df_all_features,
    age_col="Age",
    id_col="Individual",
    prop_suffix="_ratio",
    test_size=0.2,
    random_state=42,
    vt_threshold=1e-5,
    k_best=8000,
    bootstrap_n=300,
    stable_threshold=0.7,
    l1_ratio_grid=[0.5, 0.7, 0.9, 0.95, 0.99],
    final_l1_ratio_grid=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1],
    cv_splits=5,
    output_dir=str(OUTPUT_DIR / "fig3/iter_model/inte_AIDA_CIMA"),
    model_name="en_model_inte_AIDA_CIMA_bootstrap.pkl"
):
    os.makedirs(output_dir, exist_ok=True)

    # =========================
    # 1. 准备 X / y
    # =========================
    print("Preparing X and y...")
    y = df_all_features[age_col].values
    X = df_all_features.drop(columns=[age_col])

    prop_cols = [col for col in X.columns if col.endswith(prop_suffix)]
    expr_cols = [col for col in X.columns if col not in prop_cols and col != id_col]

    X_prop = X[prop_cols].copy()
    X_expr = X[expr_cols].copy()

    # =========================
    # 2. 划分训练集 / 验证集
    # =========================
    print("Splitting train and test sets...")
    X_prop_train, X_prop_test, X_expr_train, X_expr_test, y_train, y_test = train_test_split(
        X_prop,
        X_expr,
        y,
        test_size=test_size,
        random_state=random_state
    )

    # =========================
    # 3. 训练集上做方差过滤 + SelectKBest
    # =========================
    print("Performing feature selection on training set...")
    vt = VarianceThreshold(threshold=vt_threshold)
    X_expr_train_vt = vt.fit_transform(X_expr_train)
    X_expr_test_vt = vt.transform(X_expr_test)

    k_real = min(k_best, X_expr_train_vt.shape[1])

    skb = SelectKBest(score_func=f_regression, k=k_real)
    X_expr_train_selected = skb.fit_transform(X_expr_train_vt, y_train)
    X_expr_test_selected = skb.transform(X_expr_test_vt)

    selected_expr_cols = X_expr_train.columns[vt.get_support()][skb.get_support()]

    # =========================
    # 4. 标准化
    # =========================
    print("Standardizing features...")
    scaler_prop = StandardScaler()
    scaler_expr = StandardScaler()

    X_prop_train_scaled = scaler_prop.fit_transform(X_prop_train)
    X_prop_test_scaled = scaler_prop.transform(X_prop_test)

    X_expr_train_scaled = scaler_expr.fit_transform(X_expr_train_selected)
    X_expr_test_scaled = scaler_expr.transform(X_expr_test_selected)

    all_cols = list(prop_cols) + list(selected_expr_cols)

    X_train_final = np.hstack([X_prop_train_scaled, X_expr_train_scaled])
    X_test_final = np.hstack([X_prop_test_scaled, X_expr_test_scaled])

    X_train_final_df = pd.DataFrame(X_train_final, columns=all_cols, index=X_prop_train.index)
    X_test_final_df = pd.DataFrame(X_test_final, columns=all_cols, index=X_prop_test.index)

    # =========================
    # 5. 初步 ElasticNetCV 选择参数
    # =========================
    print("Performing initial ElasticNetCV for parameter tuning...")
    base_model = ElasticNetCV(
        l1_ratio=l1_ratio_grid,
        cv=cv_splits,
        max_iter=5000,
        n_jobs=-1,
        random_state=random_state
    )

    base_model.fit(X_train_final_df, y_train)

    best_alpha = base_model.alpha_
    best_l1 = base_model.l1_ratio_

    print("Best alpha:", best_alpha)
    print("Best l1_ratio:", best_l1)

    # =========================
    # 6. Bootstrap 特征稳定性筛选
    # =========================
    print("Performing bootstrap for feature stability selection...")
    feature_counter = Counter()

    for i in range(bootstrap_n):
        X_boot, y_boot = resample(
            X_train_final_df,
            y_train,
            replace=True,
            n_samples=len(y_train),
            random_state=i
        )

        model = ElasticNet(
            alpha=best_alpha,
            l1_ratio=best_l1,
            max_iter=5000,
            random_state=random_state
        )

        model.fit(X_boot, y_boot)

        selected = X_train_final_df.columns[model.coef_ != 0]

        for f in selected:
            feature_counter[f] += 1

    print("Bootstrap finished.")

    selection_freq = pd.DataFrame({
        "Feature": list(feature_counter.keys()),
        "Frequency": [feature_counter[f] / bootstrap_n for f in feature_counter]
    }).sort_values("Frequency", ascending=False)

    selection_freq.to_csv(
        os.path.join(output_dir, "bootstrap_selection_frequency.csv"),
        index=False
    )

    # =========================
    # 8. 获取稳定特征
    # =========================
    stable_features = selection_freq[
        selection_freq["Frequency"] >= stable_threshold
    ].copy()

    selected_features = stable_features["Feature"].tolist()

    if len(selected_features) == 0:
        raise ValueError(
            f"没有特征达到 stable_threshold={stable_threshold}，建议降低阈值，比如 0.5 或 0.6。"
        )

    X_train_selected = X_train_final_df[selected_features]
    X_test_selected = X_test_final_df[selected_features]

    stable_features.to_csv(
        os.path.join(output_dir, "stable_features.csv"),
        index=False
    )

    print(f"Stable features number: {len(selected_features)}")

    # =========================
    # 9. 用稳定特征重新训练最终模型
    # =========================
    print("Training final model with stable features...")
    cv = KFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=random_state
    )

    model_reduced = ElasticNetCV(
        l1_ratio=final_l1_ratio_grid,
        alphas=None,
        cv=cv,
        max_iter=5000,
        n_jobs=-1,
        random_state=random_state
    )

    scores_reduced = cross_val_score(
        model_reduced,
        X_train_selected,
        y_train,
        cv=cv,
        scoring="r2"
    )

    print(
        f"训练集交叉验证 R²："
        f"{scores_reduced.mean():.3f} ± {scores_reduced.std():.3f}"
    )

    model_reduced.fit(X_train_selected, y_train)

    # =========================
    # 10. 训练集和验证集预测
    # =========================
    print("Evaluating final model on train and test sets...")
    y_train_pred = model_reduced.predict(X_train_selected)
    y_test_pred = model_reduced.predict(X_test_selected)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    train_r, train_p = pearsonr(y_train, y_train_pred)
    test_r, test_p = pearsonr(y_test, y_test_pred)

    metrics = {
        "best_alpha_initial": best_alpha,
        "best_l1_ratio_initial": best_l1,
        "final_alpha": model_reduced.alpha_,
        "final_l1_ratio": model_reduced.l1_ratio_,
        "n_stable_features": len(selected_features),
        "cv_r2_mean_train": scores_reduced.mean(),
        "cv_r2_std_train": scores_reduced.std(),
        "train_r2": train_r2,
        "train_mae": train_mae,
        "train_pearson_r": train_r,
        "train_pearson_p": train_p,
        "test_r2": test_r2,
        "test_mae": test_mae,
        "test_pearson_r": test_r,
        "test_pearson_p": test_p
    }

    metrics_df = pd.DataFrame([metrics])
    metrics_df.to_csv(os.path.join(output_dir, "model_metrics.csv"), index=False)

    print("Train R²:", round(train_r2, 3))
    print("Test R²:", round(test_r2, 3))
    print("Train MAE:", round(train_mae, 3))
    print("Test MAE:", round(test_mae, 3))
    print("Test Pearson r:", round(test_r, 3))


    # =========================
    # 12. 训练集 + 验证集画到同一张图
    # =========================
    print("Plotting train and test predictions...")

    def plot_prediction_train_test(
        y_train,
        y_train_pred,
        y_test,
        y_test_pred,
        title=None,
        test_label="Validation",
        save_path=None,
        show=True
    ):
        import numpy as np
        import matplotlib.pyplot as plt

        from scipy.stats import pearsonr
        from sklearn.metrics import r2_score, mean_absolute_error

        # =========================
        # 1. 数据整理
        # =========================
        y_train = np.asarray(y_train).flatten()
        y_train_pred = np.asarray(y_train_pred).flatten()
        y_test = np.asarray(y_test).flatten()
        y_test_pred = np.asarray(y_test_pred).flatten()

        # 去除无效值
        train_mask = np.isfinite(y_train) & np.isfinite(y_train_pred)
        test_mask = np.isfinite(y_test) & np.isfinite(y_test_pred)

        y_train = y_train[train_mask]
        y_train_pred = y_train_pred[train_mask]

        y_test = y_test[test_mask]
        y_test_pred = y_test_pred[test_mask]

        # =========================
        # 2. 计算评价指标
        # =========================
        train_r2 = r2_score(y_train, y_train_pred)
        train_mae = mean_absolute_error(y_train, y_train_pred)
        train_mad = np.median(np.abs(y_train - y_train_pred))
        train_pcc = pearsonr(y_train, y_train_pred)[0]

        test_r2 = r2_score(y_test, y_test_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        test_mad = np.median(np.abs(y_test - y_test_pred))
        test_pcc = pearsonr(y_test, y_test_pred)[0]

        # =========================
        # 3. 论文风格基础设置
        # =========================
        train_color = "#4C78A8"      # 蓝色
        test_color = "#E76F51"       # 橙红
        ideal_color = "#8C8C8C"      # 灰色 y=x
        regression_color = "#CC4C4C" # 深红回归线

        with plt.rc_context({
            "font.family": "Arial",
            "font.size": 10,
            "axes.labelsize": 11,
            "axes.titlesize": 12,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 8.5,
            "axes.linewidth": 0.8,
            "xtick.major.width": 0.8,
            "ytick.major.width": 0.8,
            "xtick.major.size": 3.5,
            "ytick.major.size": 3.5,
            "pdf.fonttype": 42,
            "ps.fonttype": 42
        }):

            # 适合论文多面板拼图的比例
            fig, ax = plt.subplots(figsize=(4.5, 4.2))

            # =========================
            # 4. 固定坐标范围（论文统一）
            # =========================
            xmin = 0
            xmax = 120

            ax.set_xlim(xmin, xmax)
            ax.set_ylim(xmin, xmax)
            ax.set_aspect("equal", adjustable="box")

            # 每20岁一个刻度
            ticks = np.arange(0, 121, 20)
            ax.set_xticks(ticks)
            ax.set_yticks(ticks)

            # =========================
            # 5. 绘制散点
            # =========================
            ax.scatter(
                y_train,
                y_train_pred,
                s=22,
                color=train_color,
                alpha=0.55,
                edgecolors="white",
                linewidths=0.25,
                label=f"Train, n={len(y_train)}",
                zorder=2,
                rasterized=True
            )

            ax.scatter(
                y_test,
                y_test_pred,
                s=27,
                color=test_color,
                alpha=0.80,
                edgecolors="white",
                linewidths=0.3,
                label=f"{test_label}, n={len(y_test)}",
                zorder=3,
                rasterized=True
            )

            # =========================
            # 6. 理想线 y=x
            # =========================
            ax.plot(
                [xmin, xmax],
                [xmin, xmax],
                linestyle="--",
                color=ideal_color,
                linewidth=1.2,
                dashes=(4, 3),
                label=r"Ideal line, $y=x$",
                zorder=1
            )

            # =========================
            # 7. Validation/Test回归线
            # =========================
            slope, intercept = np.polyfit(
                y_test,
                y_test_pred,
                deg=1
            )

            x_line = np.linspace(xmin, xmax, 300)
            y_line = slope * x_line + intercept

            ax.plot(
                x_line,
                y_line,
                color=regression_color,
                linewidth=1.8,
                label=f"{test_label} regression",
                zorder=4
            )

            # =========================
            # 8. 指标文本
            # =========================
            metric_text = (
                r"$\bf{Train}$" + "\n"
                + rf"$R^2$ = {train_r2:.2f}" + "\n"
                + f"MAE = {train_mae:.2f}" + "\n"
                + f"MAD = {train_mad:.2f}" + "\n"
                + f"PCC = {train_pcc:.2f}" + "\n\n"
                + rf"$\bf{{{test_label}}}$" + "\n"
                + rf"$R^2$ = {test_r2:.2f}" + "\n"
                + f"MAE = {test_mae:.2f}" + "\n"
                + f"MAD = {test_mad:.2f}" + "\n"
                + f"PCC = {test_pcc:.2f}"
            )

            ax.text(
                0.04,
                0.96,
                metric_text,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8.5,
                linespacing=1.15,
                bbox=dict(
                    boxstyle="round,pad=0.35",
                    facecolor="white",
                    edgecolor="#B5B5B5",
                    linewidth=0.6,
                    alpha=0.92
                ),
                zorder=10
            )

            # =========================
            # 9. 坐标轴与标题
            # =========================
            ax.set_xlabel("Chronological age (years)")
            ax.set_ylabel("Predicted age (years)")

            if title is not None:
                ax.set_title(
                    title,
                    fontsize=12,
                    fontweight="bold",
                    pad=8
                )

            # 去掉顶部和右侧边框
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

            ax.spines["left"].set_color("#333333")
            ax.spines["bottom"].set_color("#333333")

            ax.tick_params(
                axis="both",
                direction="out",
                colors="#333333"
            )

            # 使用很弱的辅助网格
            ax.grid(
                True,
                linestyle="-",
                linewidth=0.4,
                alpha=0.16,
                color="#808080"
            )

            ax.set_axisbelow(True)

            # =========================
            # 10. 图例
            # =========================
            legend = ax.legend(
                loc="lower right",
                frameon=False,
                handlelength=2.3,
                handletextpad=0.6,
                borderaxespad=0.6,
                labelspacing=0.5
            )

            plt.tight_layout(pad=0.6)

            # =========================
            # 11. 保存
            # =========================
            if save_path is not None:
                fig.savefig(
                    os.path.join(save_path, "ElasticNet_train_validation.png"),
                    dpi=600,
                    bbox_inches="tight",
                    facecolor="white"
                )
                
                fig.savefig(
                    os.path.join(save_path, "ElasticNet_train_validation.pdf"),
                    dpi=600,
                    bbox_inches="tight",
                    facecolor="white"
                )

            if show:
                plt.show()
            else:
                plt.close(fig)

        # 返回评价指标，方便后续整理结果表
        metrics = {
            "Train_R2": train_r2,
            "Train_MAE": train_mae,
            "Train_MAD": train_mad,
            "Train_PCC": train_pcc,
            f"{test_label}_R2": test_r2,
            f"{test_label}_MAE": test_mae,
            f"{test_label}_MAD": test_mad,
            f"{test_label}_PCC": test_pcc,
            f"{test_label}_Slope": slope,
            f"{test_label}_Intercept": intercept
        }

        return fig, ax, metrics

    fig, ax, prediction_metrics = plot_prediction_train_test(
        y_train=y_train,
        y_train_pred=y_train_pred,
        y_test=y_test,
        y_test_pred=y_test_pred,
        title=None,
        test_label="Validation",
        save_path=output_dir
)

    # =========================
    # 13. 保存最终模型和预处理器
    # =========================
    print("Saving final model and preprocessors...")
    joblib.dump(
        {
            "vt": vt,
            "skb": skb,
            "scaler_prop": scaler_prop,
            "scaler_expr": scaler_expr,
            "model": model_reduced,
            "prop_cols": prop_cols,
            "expr_cols": expr_cols,
            "selected_expr_cols": list(selected_expr_cols),
            "selected_features": selected_features,
            "stable_features": stable_features,
            "metrics": metrics
        },
        os.path.join(output_dir, model_name)
    )

    print(f"Model saved to: {os.path.join(output_dir, model_name)}")

    return {
        "model": model_reduced,
        "metrics": metrics_df,
        "selection_freq": selection_freq,
        "stable_features": stable_features,
        "selected_features": selected_features,
        "X_train_selected": X_train_selected,
        "X_test_selected": X_test_selected,
        "y_train": y_train,
        "y_test": y_test,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred
    }

In [ ]:
result = build_aging_clock_elasticnet_bootstrap(
    df_all_features=df_all_features,
    age_col="Age",
    id_col="Individual",
    prop_suffix="_ratio",
    test_size=0.2,
    bootstrap_n=300,
    stable_threshold=0.7,
    output_dir=str(OUTPUT_DIR / "fig3/iter_model/inte_AIDA_CIMA")
)

## Full-cohort final fit (Fig. 3e; Extended Data Fig. 2)

Source: cells 50–53, 55, 57–59. Cell 55 intentionally reads the original saved stable-feature table. The preceding bootstrap was rerun in the research notebook for visualization; it does not replace that table. **173 selected input features remain in their original order; one fitted coefficient is zero (`CD4Tcm_CD27_SESN3`), leaving 172 nonzero features for interpretation.** Never remove the zero-coefficient input from the predictor. The full-cohort fit is an in-sample fit, distinct from external validation. This section has not been rerun.

In [ ]:
df_all_features = joblib.load(str(FEATURES_1000))
df_all_features

In [ ]:
# 目标变量 y 和特征矩阵 X
y = df_all_features['Age'].values
X = df_all_features.drop(columns='Age')

# 区分细胞比例特征（假设以 "_prop" 结尾）和表达特征（其余）
prop_cols = [col for col in X.columns if col.endswith('_ratio')]
expr_cols = [col for col in X.columns if col not in prop_cols and col != 'Individual']
expr_cols

X_prop = X[prop_cols].copy()
X_expr = X[expr_cols].copy()

# 去除低方差表达特征
vt = VarianceThreshold(threshold=1e-5)
X_expr_vt = vt.fit_transform(X_expr)

# 选择最相关的前 k 个特征
skb = SelectKBest(score_func=f_regression, k=8000)   # k=8000
X_expr_selected = skb.fit_transform(X_expr_vt, y)

# 获取保留下来的列名
selected_expr_cols = X_expr.columns[vt.get_support()][skb.get_support()]

# 合并，标准化
scaler_prop = StandardScaler()
scaler_expr = StandardScaler()

X_prop_scaled = scaler_prop.fit_transform(X_prop)
X_expr_scaled = scaler_expr.fit_transform(X_expr_selected)

# 合并，带列名
all_cols = list(X_prop.columns) + list(selected_expr_cols)
X_final = np.hstack([X_prop_scaled, X_expr_scaled])
X_final_df = pd.DataFrame(X_final, columns=all_cols)
X_final_df

In [ ]:
# 选择最好的参数
from sklearn.linear_model import ElasticNetCV

base_model = ElasticNetCV(
    l1_ratio=[0.5, 0.7, 0.9, 0.95, 0.99],
    cv=5,
    max_iter=5000,
    n_jobs=-1
)

base_model.fit(X_final_df, y)

best_alpha = base_model.alpha_
best_l1 = base_model.l1_ratio_

print("Best alpha:", best_alpha)
print("Best l1_ratio:", best_l1)

In [ ]:
# bootstrap 计算频率
from sklearn.linear_model import ElasticNet
from sklearn.utils import resample
from collections import Counter
import pandas as pd
import numpy as np

n_boot = 300   # 200 已经足够稳定
feature_counter = Counter()

for i in range(n_boot):
    
    X_boot, y_boot = resample(
        X_final_df,
        y,
        replace=True,
        n_samples=len(y),
        random_state=i
    )
    
    model = ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1,
        max_iter=5000
    )
    
    model.fit(X_boot, y_boot)
    
    selected = X_final_df.columns[model.coef_ != 0]
    
    for f in selected:
        feature_counter[f] += 1

print("Bootstrap finished.")
# 计算选择频率
selection_freq = pd.DataFrame({
    "Feature": list(feature_counter.keys()),
    "Frequency": [feature_counter[f]/n_boot for f in feature_counter]
})

selection_freq = selection_freq.sort_values("Frequency", ascending=False)

In [ ]:
# 上面是为了画图又做了一遍bootstrap，下面直接用已经筛选好的特征
stable_features = pd.read_csv(str(FINAL_STABLE_FEATURES))
selected_features = stable_features['Feature'].tolist()
X_selected = X_final_df[selected_features]

# 3. 定义交叉验证
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 4. 训练模型并评估
model_reduced = ElasticNetCV(
    l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
    alphas=None,
    cv=cv,
    max_iter=5000,
    n_jobs=-1
)

# 交叉验证性能
scores_reduced = cross_val_score(model_reduced, X_selected, y, cv=cv, scoring='r2')
print(f"使用stable的173特征的交叉验证 R² 得分（平均±标准差）：{scores_reduced.mean():.3f} ± {scores_reduced.std():.3f}")


In [ ]:
model_reduced.fit(X_selected, y)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr


def plot_final_en_prediction(
    y_true,
    y_pred,
    title=None,
    save_path=None,
    show=True
):
    """
    绘制最终 ElasticNet 模型的整体拟合效果。
    使用 hexbin 展示样本密度，与训练/验证散点图区分。
    """

    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) < 2:
        raise ValueError("有效样本数少于2，无法绘图。")

    # 评价指标
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mad = np.median(np.abs(y_true - y_pred))
    pcc, p_value = pearsonr(y_true, y_pred)

    # 回归线
    slope, intercept = np.polyfit(y_true, y_pred, 1)
    x_line = np.linspace(0, 120, 300)
    y_line = slope * x_line + intercept

    with plt.rc_context({
        "font.family": "Arial",
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.linewidth": 0.8,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }):
        fig, ax = plt.subplots(figsize=(4.7, 4.2))

        # 样本密度图
        hb = ax.hexbin(
            y_true,
            y_pred,
            gridsize=32,
            extent=(0, 120, 0, 120),
            mincnt=1,
            cmap="Blues",
            linewidths=0.25,
            edgecolors="white",
            bins="log"
        )

        # y=x理想线
        ax.plot(
            [0, 120],
            [0, 120],
            linestyle="--",
            color="#6E6E6E",
            linewidth=1.2,
            dashes=(4, 3),
            label=r"Ideal line, $y=x$"
        )

        # 最终模型回归线
        ax.plot(
            x_line,
            y_line,
            color="#D55E00",
            linewidth=1.8,
            label="EN regression"
        )

        # 指标文字，不使用大白框
        metric_text = (
            rf"$R^2$ = {r2:.2f}" + "\n"
            + f"MAE = {mae:.2f}" + "\n"
            + f"MAD = {mad:.2f}" + "\n"
            + f"PCC = {pcc:.2f}" + "\n"
            + f"n = {len(y_true)}"
        )

        ax.text(
            0.04,
            0.96,
            metric_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.8,
            linespacing=1.2,
            color="#222222"
        )

        ax.set_xlim(0, 120)
        ax.set_ylim(0, 120)
        ax.set_xticks(np.arange(0, 121, 20))
        ax.set_yticks(np.arange(0, 121, 20))
        ax.set_aspect("equal", adjustable="box")

        ax.set_xlabel("Chronological age (years)")
        ax.set_ylabel("Predicted age (years)")

        if title is not None:
            ax.set_title(title, fontweight="bold", pad=8)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", direction="out")

        ax.grid(True, linewidth=0.4, alpha=0.12)
        ax.set_axisbelow(True)

        ax.legend(
            loc="lower right",
            frameon=False,
            fontsize=8.5,
            handlelength=2.3
        )

        # 密度色条
        cbar = fig.colorbar(hb, ax=ax, pad=0.025, fraction=0.045)
        cbar.set_label("Sample density", fontsize=9)
        cbar.ax.tick_params(labelsize=8, width=0.6)
        cbar.outline.set_linewidth(0.6)

        fig.tight_layout(pad=0.6)

        if save_path is not None:
            save_dir = os.path.dirname(save_path)
            if save_dir:
                os.makedirs(save_dir, exist_ok=True)

            fig.savefig(save_path, dpi=600, bbox_inches="tight", facecolor="white")

        if show:
            plt.show()

        plt.close(fig)

    metrics = {
        "R2": r2,
        "MAE": mae,
        "MAD": mad,
        "PCC": pcc,
        "PCC_pvalue": p_value,
        "Slope": slope,
        "Intercept": intercept,
        "n": len(y_true)
    }

    return fig, ax, metrics

In [ ]:
### 可视化在训练集上的拟合效果（plot_en_model函数在上个部分）

from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# 确保 model 已训练，且 X_final 和 y 已准备好
# model_reduced = pipeline['model']
y_pred = model_reduced.predict(X_selected)

fig, ax, final_metrics = plot_final_en_prediction(
    y_true=y,
    y_pred=y_pred,
    title=None,
    save_path=str(OUTPUT_DIR / "fig3/final_model/ElasticNet_final_fit.pdf")
)